# 8주차 A회차 실습 과제: 뉴스 기사 토픽 모델링

이 노트북은 `docs/ch8A.md`의 코딩 시연을 그대로 옮긴 것이 아니라, 하나의 과제를 처음부터 끝까지 수행하도록 구성한 실습용 노트북이다.

## 과제

가상의 한국어 뉴스 기사 묶음에서 BERTopic으로 숨겨진 주제를 찾고, 각 주제의 핵심 단어와 대표 문서를 근거로 주제 이름을 붙인다.

## 산출물

- `data/output/ch8A_topic_info.csv`: 발견된 주제 요약
- `data/output/ch8A_document_topics.csv`: 문서별 주제 배정 결과
- `data/output/ch8A_topic_keywords.json`: 주제별 핵심 단어
- 주제별 핵심 단어 시각화: 셀에 직접 출력
- 주제 간 유사도 시각화: 셀에 직접 출력
- 문서 임베딩 지도: 셀에 직접 출력
- 주제 계층 구조 / Topic Network: 셀에 직접 출력
- 단어 순위 시각화: 셀에 직접 출력
- 특정 문서의 주제 분포: 셀에 직접 출력
- `data/output/ch8A_topics_over_time_result.csv`: 시간별 주제 빈도
- Dynamic Topic Modeling 시각화: 셀에 직접 출력


## 0. 환경 확인

프로젝트 루트에서 `setup_env.py`를 실행하면 `.venv`가 생성되고 기본 패키지가 설치된다. Jupyter에서는 `2026NLP` 커널을 선택한 뒤 이 노트북을 실행한다.

macOS/Linux:

```bash
cd /path/to/2026NLP
python setup_env.py
source .venv/bin/activate
python -m ipykernel install --user --name 2026nlp --display-name "2026NLP"
```

Windows PowerShell:

```powershell
cd C:\path\to\2026NLP
python setup_env.py
.\.venv\Scripts\Activate.ps1
python -m ipykernel install --user --name 2026nlp --display-name "2026NLP"
```


In [ ]:
# ============================================================
# 루트 가상환경 및 chapter8 패키지 확인
# ============================================================
from pathlib import Path
import importlib.util
import subprocess
import sys


def find_project_root(start: Path) -> Path:
    """현재 위치에서 위로 올라가며 2026NLP 프로젝트 루트를 찾는다."""
    for candidate in [start, *start.parents]:
        if (candidate / "setup_env.py").exists() and (candidate / "practice" / "chapter8").exists():
            return candidate
    raise RuntimeError(
        "프로젝트 루트를 찾을 수 없습니다. "
        "노트북을 2026NLP 프로젝트 안에서 실행하세요."
    )


ROOT_DIR = find_project_root(Path.cwd().resolve())
CHAPTER_DIR = ROOT_DIR / "practice" / "chapter8"
ROOT_VENV = ROOT_DIR / ".venv"
REQUIREMENTS = CHAPTER_DIR / "requirements.txt"

if Path(sys.prefix).resolve() != ROOT_VENV.resolve():
    raise RuntimeError(
        "현재 노트북 커널이 루트 .venv가 아닙니다. "
        "프로젝트 루트에서 `python setup_env.py`를 실행한 뒤 Jupyter 커널을 `2026NLP`로 선택하세요.\n"
        f"현재 Python: {sys.executable}\n"
        f"필요한 가상환경: {ROOT_VENV}"
    )

required_modules = {
    "bertopic": "bertopic",
    "sentence_transformers": "sentence-transformers",
    "umap": "umap-learn",
    "hdbscan": "hdbscan",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "plotly": "plotly",
    "nbformat": "nbformat",
    "kiwipiepy": "kiwipiepy",
}

missing = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    print(f"설치되지 않은 chapter8 패키지: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS)])
else:
    print("chapter8 패키지 확인 완료")

print(f"프로젝트 루트: {ROOT_DIR}")
print(f"사용 중인 Python: {sys.executable}")
print(f"가상환경: {sys.prefix}")


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    import torch
    from bertopic import BERTopic
    from hdbscan import HDBSCAN
    from sentence_transformers import SentenceTransformer
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    from kiwipiepy import Kiwi
    from umap import UMAP
except ImportError as exc:
    raise ImportError(
        "필수 패키지가 설치되어 있지 않습니다. "
        "루트에서 `python -m pip install -r practice/chapter8/requirements.txt`를 실행하세요."
    ) from exc

SEED = 42
np.random.seed(SEED)

try:
    CHAPTER_DIR
except NameError:
    if "find_project_root" not in globals():
        def find_project_root(start: Path) -> Path:
            for candidate in [start, *start.parents]:
                if (candidate / "setup_env.py").exists() and (candidate / "practice" / "chapter8").exists():
                    return candidate
            raise RuntimeError("노트북을 2026NLP 프로젝트 안에서 실행하세요.")

    ROOT_DIR = find_project_root(Path.cwd().resolve())
    CHAPTER_DIR = ROOT_DIR / "practice" / "chapter8"

BASE_DIR = CHAPTER_DIR
OUTPUT_DIR = BASE_DIR / "data" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"실행 디바이스: {device}")
print(f"산출물 저장 위치: {OUTPUT_DIR}")


## 1. 과제 데이터 로드

`data/input/ch8A_topics_over_time_input.csv`에 미리 준비된 시계열 뉴스 데이터를 불러온다. 이 데이터는 처음부터 Dynamic Topic Modeling을 염두에 두고 `date`, `text`, `true_topic` 컬럼을 포함한다. `date`는 시간별 주제 변화 분석에 사용한다. 한국어 키워드 품질을 높이기 위해 Kiwi로 명사만 추출한 `text_nouns`를 topic representation에 사용하고, `true_topic`은 모델 학습에는 사용하지 않으며 마지막 검증 단계에서만 비교한다.


In [ ]:
INPUT_PATH = BASE_DIR / "data" / "input" / "ch8A_topics_over_time_input.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"과제 데이터 파일을 찾을 수 없습니다: {INPUT_PATH}")

df = pd.read_csv(INPUT_PATH)
required_columns = {"date", "text", "true_topic"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"데이터에 필요한 컬럼이 없습니다: {sorted(missing_columns)}")

df["date"] = pd.to_datetime(df["date"])
df = df.dropna(subset=["date", "text"]).sort_values("date").reset_index(drop=True)
docs_raw = df["text"].astype(str).tolist()
timestamps = df["date"].tolist()

kiwi = Kiwi()
noun_tags = {"NNG", "NNP", "SL"}
stop_nouns = {
    "뉴스", "기자", "이번", "관련", "당국", "전문가",
    "국내", "발표", "개편", "개선", "확대", "강화", "절차", "기준",
    "기업", "연구", "문제", "결과", "공개", "서비스", "업무",
}

def extract_nouns(text):
    nouns = []
    for token in kiwi.tokenize(text):
        if token.tag in noun_tags and len(token.form) >= 2 and token.form not in stop_nouns:
            nouns.append(token.form)
    return " ".join(nouns)

df["text_nouns"] = [extract_nouns(text) for text in docs_raw]
docs = df["text_nouns"].tolist()

empty_docs = df[df["text_nouns"].str.len() == 0]
if len(empty_docs) > 0:
    raise ValueError(f"명사 추출 결과가 비어 있는 문서가 있습니다: {empty_docs.index.tolist()}")

print(f"데이터 파일: {INPUT_PATH}")
print(f"문서 수: {len(docs_raw)}")
print(f"기간: {df['date'].min().date()} ~ {df['date'].max().date()}")
print("주제별 문서 수:")
print(df["true_topic"].value_counts().sort_index())
display(df[["date", "text", "text_nouns", "true_topic"]].head(8))


## 2. BERTopic 모델 구성과 학습

파이프라인은 `원문 문서 임베딩 → UMAP 차원 축소 → HDBSCAN 클러스터링 → 명사 기반 c-TF-IDF 주제 표현` 순서로 실행된다.


In [ ]:
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)

umap_model = UMAP(
    n_neighbors=5,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)

hdbscan_model = HDBSCAN(
    # 실습 데이터는 주제별 12개 문서로 작으므로 4로 낮춰
    # 건강/기술처럼 가까운 주제가 축소 과정에서 합쳐지는 것을 줄인다.
    min_cluster_size=4,
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    token_pattern=r"(?u)[가-힣A-Za-z0-9]{2,}",
    min_df=1,
    ngram_range=(1, 1),
    stop_words=list(stop_nouns),
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    nr_topics=5,
    verbose=False,
)

embeddings = embedding_model.encode(docs_raw, show_progress_bar=True)
topics, probabilities = topic_model.fit_transform(docs, embeddings=embeddings)
df["topic"] = topics

topic_info = topic_model.get_topic_info()
display(topic_info)

n_topics = len([t for t in set(topics) if t != -1])
n_noise = sum(1 for t in topics if t == -1)
print(f"발견된 주제 수: {n_topics}")
print(f"노이즈 문서 수: {n_noise}")


## 3. 주제별 핵심 단어와 대표 문서 확인

아래 결과를 보고 각 주제가 무엇을 의미하는지 직접 이름을 붙인다.


In [ ]:
representative_docs_by_topic = topic_model.get_representative_docs()
topic_keywords = {}
topic_keyword_scores = {}
valid_topic_ids = sorted(t for t in df["topic"].unique() if t != -1)
grouped_texts = [" ".join(df.loc[df["topic"] == topic_id, "text_nouns"]) for topic_id in valid_topic_ids]

keyword_vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)[가-힣A-Za-z0-9]{2,}",
    min_df=1,
    ngram_range=(1, 1),
    max_features=500,
)
topic_term_matrix = keyword_vectorizer.fit_transform(grouped_texts)
terms = keyword_vectorizer.get_feature_names_out()

for row_idx, topic_id in enumerate(valid_topic_ids):
    scores = topic_term_matrix[row_idx].toarray().ravel()
    top_indices = scores.argsort()[::-1][:8]

    keyword_pairs = [(terms[i], float(scores[i])) for i in top_indices if scores[i] > 0]
    keywords = [word for word, score in keyword_pairs]
    topic_keywords[int(topic_id)] = keywords
    topic_keyword_scores[int(topic_id)] = keyword_pairs

    representative_docs = representative_docs_by_topic.get(topic_id, [])[:2]
    print("=" * 80)
    print(f"주제 {topic_id}")
    print(f"핵심 단어: {', '.join(keywords)}")
    print("대표 문서:")
    for doc in representative_docs:
        print(f"- {doc}")

with open(OUTPUT_DIR / "ch8A_topic_keywords.json", "w", encoding="utf-8") as f:
    json.dump(topic_keywords, f, ensure_ascii=False, indent=2)


## 4. 실제 주제와 모델 주제 비교

실제 수업에서는 정답 레이블이 없는 경우가 많다. 여기서는 실습 검증을 위해 생성 시 사용한 `true_topic`과 BERTopic의 `topic`을 비교한다.


In [ ]:
comparison = pd.crosstab(df["topic"], df["true_topic"], margins=True)
display(comparison)

topic_label_guess = {}
topic_validation_rows = []
for topic_id in sorted(t for t in df["topic"].unique() if t != -1):
    topic_docs = df[df["topic"] == topic_id]
    label_counts = topic_docs["true_topic"].value_counts()
    majority_label = label_counts.idxmax()
    purity = label_counts.max() / len(topic_docs)
    topic_label_guess[int(topic_id)] = majority_label
    topic_validation_rows.append({
        "topic": int(topic_id),
        "estimated_label": majority_label,
        "documents": len(topic_docs),
        "purity": round(purity, 3),
        "keywords": ", ".join(topic_keywords.get(int(topic_id), [])[:5]),
    })

validation_df = pd.DataFrame(topic_validation_rows).sort_values("topic")
display(validation_df)

weighted_purity = sum(row["documents"] * row["purity"] for row in topic_validation_rows) / max(len(df[df["topic"] != -1]), 1)
noise_count = int((df["topic"] == -1).sum())
print("\n[토픽 예측 결과 해석]")
print(f"- 모델은 노이즈를 제외하고 {len(topic_validation_rows)}개 토픽을 찾았다.")
print(f"- 가중 평균 순도는 {weighted_purity:.3f}이다. 1.0에 가까울수록 실제 주제와 모델 토픽이 잘 대응한다.")
print(f"- 노이즈(-1) 문서는 {noise_count}개이다.")

if weighted_purity >= 0.95 and noise_count == 0:
    print("- 현재 결과는 실습 데이터의 실제 주제와 거의 일대일로 대응하므로, 토픽 모델링 결과를 해석하기 좋은 상태다.")
elif weighted_purity >= 0.80:
    print("- 대체로 잘 분리되었지만 일부 주제가 섞였다. 섞인 토픽의 키워드와 대표 문서를 추가로 확인해야 한다.")
else:
    print("- 주제 혼합이 큰 편이다. 데이터 생성 방식, stopword, HDBSCAN/UMAP 설정을 다시 점검해야 한다.")

for row in topic_validation_rows:
    print(
        f"- 토픽 {row['topic']}은 '{row['estimated_label']}'로 해석할 수 있다 "
        f"(문서 {row['documents']}개, 순도 {row['purity']:.2f}, 핵심어: {row['keywords']})."
    )


## 5. 특정 문서의 주제 분석

문서 하나를 골라 모델이 어떤 주제로 분류했는지 확인하고, 분류가 타당한지 해석한다.


In [ ]:
doc_idx = 0
doc_topic = int(df.loc[doc_idx, "topic"])
doc_true_topic = df.loc[doc_idx, "true_topic"]
doc_text = df.loc[doc_idx, "text"]
doc_nouns = df.loc[doc_idx, "text_nouns"]

print(f"문서 번호: {doc_idx}")
print(f"문서 날짜: {df.loc[doc_idx, 'date'].date()}")
print(f"문서 내용: {doc_text}")
print(f"명사 추출: {doc_nouns}")
print(f"실제 주제: {doc_true_topic}")
print(f"모델 주제: {doc_topic}")
print(f"모델 주제 이름 추정: {topic_label_guess.get(doc_topic, '노이즈')}")

doc_distribution_df = pd.DataFrame()
if probabilities is not None:
    probs = np.asarray(probabilities)
    if probs.ndim == 2 and doc_idx < len(probs):
        probability_values = probs[doc_idx]
        probability_topics = valid_topic_ids[: len(probability_values)]
        doc_distribution_df = pd.DataFrame({
            "topic": probability_topics,
            "estimated_label": [topic_label_guess.get(int(t), f"주제 {t}") for t in probability_topics],
            "probability": probability_values,
        }).sort_values("probability", ascending=False)
        display(doc_distribution_df)

        top_row = doc_distribution_df.iloc[0]
        second_prob = float(doc_distribution_df.iloc[1]["probability"]) if len(doc_distribution_df) > 1 else 0.0
        confidence = float(top_row["probability"])
        margin = confidence - second_prob
        predicted_label = str(top_row["estimated_label"])
        is_correct = predicted_label == str(doc_true_topic)

        print("\n[단일 문서 토픽 예측 해석]")
        print(f"- 가장 높은 확률의 예측 주제는 '{predicted_label}'이며 확률은 {confidence:.3f}이다.")
        print(f"- 1위와 2위 주제의 확률 차이는 {margin:.3f}이다. 차이가 클수록 모델이 더 확신한다.")
        print(f"- 실제 주제와 비교하면 {'일치한다' if is_correct else '일치하지 않는다'}.")
        if is_correct and margin >= 0.30:
            print("- 이 문서는 핵심 단어와 확률 분포가 모두 예측 주제를 강하게 지지한다.")
        elif is_correct:
            print("- 예측은 맞지만 2위 주제와 차이가 크지 않다. 문장 안에 다른 주제와 겹치는 단어가 있는지 확인한다.")
        else:
            print("- 예측이 빗나갔다면 명사 추출 결과와 해당 토픽의 대표 문서를 비교해 혼동 원인을 찾는다.")

        import plotly.express as px
        fig = px.bar(
            doc_distribution_df,
            x="estimated_label",
            y="probability",
            text="probability",
            title=f"문서 {doc_idx}의 주제 분포",
        )
        fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
        fig.update_layout(xaxis_title="주제", yaxis_title="확률", yaxis_range=[0, 1.05])
        display(fig)
    elif probs.ndim == 1 and doc_idx < len(probs):
        print(f"\n배정 신뢰도: {probs[doc_idx]:.3f}")


## 6. 시각화 출력

Plotly 기반 시각화는 노트북 셀 아래에 직접 출력되어 상호작용하며 확인할 수 있다.


In [ ]:
visualization_outputs = []

try:
    import plotly.express as px

    keyword_rows = []
    for topic_id, pairs in topic_keyword_scores.items():
        label = topic_label_guess.get(topic_id, f"주제 {topic_id}")
        for word, score in pairs:
            keyword_rows.append({"topic": f"주제 {topic_id}: {label}", "word": word, "score": score})

    keyword_df = pd.DataFrame(keyword_rows)
    fig = px.bar(
        keyword_df,
        x="score",
        y="word",
        color="topic",
        facet_col="topic",
        facet_col_wrap=2,
        orientation="h",
        height=900,
        title="주제별 핵심 단어 TF-IDF 점수",
    )
    fig.update_yaxes(matches=None, autorange="reversed")
    fig.update_xaxes(matches=None)
    fig.update_layout(showlegend=False)
    display(fig)
    visualization_outputs.append("주제별 핵심 단어 시각화")
except Exception as exc:
    print(f"barchart 출력 실패: {exc}")

try:
    fig = topic_model.visualize_heatmap(top_n_topics=min(8, n_topics))
    display(fig)
    visualization_outputs.append("주제 간 유사도 heatmap")
except Exception as exc:
    print(f"heatmap 출력 실패: {exc}")


try:
    fig = topic_model.visualize_hierarchy()
    display(fig)
    visualization_outputs.append("주제 계층 구조")
except Exception as exc:
    print(f"BERTopic hierarchy 생성 실패, 주제 유사도 네트워크로 대체합니다: {exc}")
    import plotly.graph_objects as go
    from sklearn.metrics.pairwise import cosine_similarity

    network_topic_ids = valid_topic_ids
    embeddings_for_network = embeddings
    centroids = np.vstack([
        embeddings_for_network[df["topic"].to_numpy() == topic_id].mean(axis=0)
        for topic_id in network_topic_ids
    ])
    similarity = cosine_similarity(centroids)

    angles = np.linspace(0, 2 * np.pi, len(network_topic_ids), endpoint=False)
    x = np.cos(angles)
    y = np.sin(angles)
    edge_x, edge_y = [], []
    for i in range(len(network_topic_ids)):
        for j in range(i + 1, len(network_topic_ids)):
            if similarity[i, j] >= 0.35:
                edge_x += [x[i], x[j], None]
                edge_y += [y[i], y[j], None]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=1, color="#999"), hoverinfo="none"))
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode="markers+text",
        text=[f"주제 {topic_id}<br>{topic_label_guess.get(int(topic_id), '')}" for topic_id in network_topic_ids],
        textposition="top center",
        marker=dict(size=28, color=list(range(len(network_topic_ids))), colorscale="Viridis", showscale=False),
        hovertemplate="%{text}<extra></extra>",
    ))
    fig.update_layout(
        title="Topic Network: 주제 중심점 간 유사도",
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        showlegend=False,
        height=650,
    )
    display(fig)
    visualization_outputs.append("주제 유사도 네트워크")

try:
    fig = topic_model.visualize_term_rank()
    display(fig)
    visualization_outputs.append("단어 순위 시각화")
except Exception as exc:
    print(f"term rank 출력 실패: {exc}")

try:
    fig = topic_model.visualize_documents(docs_raw, embeddings=embeddings)
    display(fig)
    visualization_outputs.append("문서 임베딩 지도")
except Exception as exc:
    print(f"documents 출력 실패: {exc}")

for name in visualization_outputs:
    print(f"셀 출력 완료: {name}")


## 7. Dynamic Topic Modeling

`date` 컬럼을 사용해 시간에 따라 각 주제의 빈도가 어떻게 변하는지 확인한다.


In [ ]:
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps,
    topics=topics,
    nr_bins=6,
)

topics_over_time_path = OUTPUT_DIR / "ch8A_topics_over_time_result.csv"
topics_over_time.to_csv(topics_over_time_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {topics_over_time_path}")
display(topics_over_time.head(12))

time_summary_rows = []
for topic_id in sorted(t for t in topics_over_time["Topic"].unique() if t != -1):
    topic_series = topics_over_time[topics_over_time["Topic"] == topic_id].sort_values("Timestamp")
    if topic_series.empty:
        continue
    peak_row = topic_series.loc[topic_series["Frequency"].idxmax()]
    first_freq = float(topic_series.iloc[0]["Frequency"])
    last_freq = float(topic_series.iloc[-1]["Frequency"])
    delta = last_freq - first_freq
    if delta > 0:
        trend = "증가"
    elif delta < 0:
        trend = "감소"
    else:
        trend = "유지"
    time_summary_rows.append({
        "topic": int(topic_id),
        "estimated_label": topic_label_guess.get(int(topic_id), f"주제 {topic_id}"),
        "keywords": ", ".join(topic_keywords.get(int(topic_id), [])[:4]),
        "first_frequency": first_freq,
        "last_frequency": last_freq,
        "change": delta,
        "trend": trend,
        "peak_time": peak_row["Timestamp"],
        "peak_frequency": float(peak_row["Frequency"]),
    })

time_summary_df = pd.DataFrame(time_summary_rows)
display(time_summary_df)

print("\n[시간별 토픽 변화 해석]")
print("- Frequency는 해당 시간 구간에서 그 토픽으로 분류된 문서 수를 뜻한다.")
if not time_summary_df.empty:
    for row in time_summary_rows:
        print(
            f"- {row['estimated_label']} 토픽은 첫 구간 {row['first_frequency']:.0f}건, "
            f"마지막 구간 {row['last_frequency']:.0f}건으로 '{row['trend']}' 패턴이다. "
            f"최고점은 {row['peak_time']}의 {row['peak_frequency']:.0f}건이다."
        )
    dominant = time_summary_df.sort_values("peak_frequency", ascending=False).iloc[0]
    print(f"- 가장 높은 순간 빈도를 보인 토픽은 '{dominant['estimated_label']}'이다.")
else:
    print("- 해석할 유효 토픽이 없다.")

try:
    fig = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=min(5, n_topics))
    display(fig)
except Exception as exc:
    print(f"dynamic topic 시각화 출력 실패: {exc}")


## 8. 결과 저장과 제출용 질문

아래 파일을 저장한 뒤, 다음 질문에 답한다.

1. BERTopic이 발견한 주제 수는 몇 개인가?
2. 각 주제의 핵심 단어와 대표 문서를 근거로 주제 이름을 어떻게 붙일 수 있는가?
3. 노이즈 문서는 몇 개이며, 노이즈가 생긴 이유는 무엇이라고 해석할 수 있는가?
4. 실제 주제와 모델 주제가 잘 맞지 않는 사례가 있다면 왜 그런가?


In [ ]:
topic_info_path = OUTPUT_DIR / "ch8A_topic_info.csv"
document_topics_path = OUTPUT_DIR / "ch8A_document_topics.csv"

topic_info.to_csv(topic_info_path, index=False, encoding="utf-8-sig")
df.to_csv(document_topics_path, index=False, encoding="utf-8-sig")

print(f"저장 완료: {topic_info_path}")
print(f"저장 완료: {document_topics_path}")
print(f"저장 완료: {OUTPUT_DIR / 'ch8A_topic_keywords.json'}")

display(df.head(10))


---

# 8주차 B회차: BERTopic 다듬기 + 동적/멀티모달 확장

여기서부터는 `docs/ch8B.md`와 짝이 되는 셀이다. 위(0~8절)에서 만든 `topic_model`, `documents`(=`docs`), `topics`, `probabilities`, `embeddings`를 그대로 이어 사용한다.


## 9. 토픽 평가지표 — NPMI · Topic Diversity

지표 한 줄 정리: **일관성**(NPMI, 높을수록 좋음, 범위 −1~1)과 **다양성**(Topic Diversity, 0~1, 높을수록 좋음)을 함께 본다. 본격적인 C_v는 외부 라이브러리(`gensim`)가 필요하므로, 여기서는 의존을 최소화한 **자체 구현** NPMI + Topic Diversity로 직관을 본다.


In [ ]:
from collections import Counter
from itertools import combinations


def topic_diversity(topic_model, top_n: int = 10) -> float:
    """모든 토픽의 상위 N개 단어 중 고유 단어 비율 (0~1, ↑ 좋음)."""
    topics_ = [t for t in topic_model.get_topics().keys() if t != -1]
    all_words: list[str] = []
    for t in topics_:
        words = [w for w, _ in topic_model.get_topic(t)[:top_n] if w]
        all_words.extend(words)
    return len(set(all_words)) / max(len(all_words), 1)


def npmi_coherence(topic_model, docs_for_eval, top_n: int = 10) -> float:
    """문서 동시 출현 기반의 NPMI 평균 (간이 구현, ↑ 좋음, 범위 −1~1)."""
    tokenized = [set(d.split()) for d in docs_for_eval]
    N = len(tokenized)
    df_count = Counter(w for d in tokenized for w in d)
    scores: list[float] = []
    topics_ = [t for t in topic_model.get_topics().keys() if t != -1]
    for t in topics_:
        words = [w for w, _ in topic_model.get_topic(t)[:top_n] if w]
        for w_i, w_j in combinations(words, 2):
            n_ij = sum(1 for d in tokenized if w_i in d and w_j in d)
            if n_ij == 0 or df_count[w_i] == 0 or df_count[w_j] == 0:
                continue
            p_ij = n_ij / N
            p_i = df_count[w_i] / N
            p_j = df_count[w_j] / N
            pmi = np.log(p_ij / (p_i * p_j))
            npmi = pmi / (-np.log(p_ij))
            scores.append(npmi)
    return float(np.mean(scores)) if scores else 0.0


eval_npmi = npmi_coherence(topic_model, docs)
eval_div = topic_diversity(topic_model)
print(f"NPMI            : {eval_npmi:+.4f}  (높을수록 좋음, −1~1)")
print(f"Topic Diversity : {eval_div:.4f}  (높을수록 좋음, 0~1)")

eval_summary = pd.DataFrame(
    [{"metric": "NPMI", "value": eval_npmi, "criteria": ">0.05 수용, >0.15 양호"},
     {"metric": "Topic Diversity", "value": eval_div, "criteria": ">0.7 양호"}]
)
display(eval_summary)
eval_summary.to_csv(OUTPUT_DIR / "ch8B_eval_metrics.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_DIR / 'ch8B_eval_metrics.csv'}")

print("\n[평가지표 해석]")
if eval_npmi > 0.15:
    print("- NPMI가 양호한 수준이다. 같은 토픽의 상위 단어들이 문서 안에서 함께 등장하는 경향이 있다.")
elif eval_npmi > 0.05:
    print("- NPMI가 수용 가능한 수준이다. 토픽 단어 조합은 어느 정도 일관성이 있다.")
else:
    print("- NPMI가 낮다. 상위 키워드가 한 주제를 안정적으로 설명하는지 대표 문서를 확인해야 한다.")

if eval_div > 0.7:
    print("- Topic Diversity가 높다. 서로 다른 토픽이 중복되지 않는 키워드로 표현되고 있다.")
else:
    print("- Topic Diversity가 낮다. 여러 토픽에서 같은 키워드가 반복되는지 stopword와 토픽 수를 점검한다.")


## 10. 노이즈 처리 — `reduce_outliers()`

HDBSCAN이 -1로 분리한 문서를 **가장 유사한 토픽**에 다시 배정한다. 노이즈가 많을 때 유효하다. 이 데이터셋은 `nr_topics=5`로 명시 지정하여 학습되었으므로 노이즈가 적을 수 있다 — 그런 경우에는 셀이 자동으로 알려주고 안전하게 통과한다.


In [ ]:
topics_array = np.asarray(topics)
n_outliers_before = int((topics_array == -1).sum())
print(f"노이즈(-1) 문서 수: {n_outliers_before} / {len(topics_array)}")

if n_outliers_before == 0:
    print("이 모델에는 노이즈가 없어 reduce_outliers()는 변화를 만들지 않는다.")
    new_topics_list = list(topics_array)
else:
    new_topics_list = topic_model.reduce_outliers(
        documents=docs,
        topics=list(topics_array),
        strategy="distributions",
    )
    n_outliers_after = sum(1 for t in new_topics_list if t == -1)
    print(f"재할당 후 남은 노이즈: {n_outliers_after}")

# 전후 비교 표
before_counts = pd.Series(topics_array).value_counts().rename("before").sort_index()
after_counts = pd.Series(new_topics_list).value_counts().rename("after").sort_index()
compare_df = pd.concat([before_counts, after_counts], axis=1).fillna(0).astype(int)
compare_df["delta"] = compare_df["after"] - compare_df["before"]
display(compare_df)
compare_df.to_csv(OUTPUT_DIR / "ch8B_outlier_reduction.csv", encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_DIR / 'ch8B_outlier_reduction.csv'}")

print("\n[노이즈 처리 해석]")
if n_outliers_before == 0:
    print("- 모든 문서가 토픽에 배정되었으므로 현재 데이터에서는 노이즈 처리보다 토픽 해석이 더 중요하다.")
else:
    moved = n_outliers_before - sum(1 for t in new_topics_list if t == -1)
    print(f"- {moved}개 노이즈 문서가 기존 토픽으로 재배정되었다.")
    print("- 재배정 후 특정 토픽의 문서 수가 과도하게 늘면, 원래 애매한 문서가 한 토픽으로 몰렸을 가능성을 확인한다.")


## 11. 토픽 축소·병합 — `reduce_topics()`

비슷한 토픽을 c-TF-IDF 코사인 유사도 기준으로 자동 병합한다. 이 데이터에서는 5개 토픽을 4개로 줄여 본다. 병합 후 키워드는 자동 재계산된다.

> **주의**: `reduce_topics()`는 토픽 모델의 내부 상태를 영구적으로 변경한다. 다음 셀에서는 이 변화를 반영한 `topic_model`이 사용된다.


In [ ]:
before_info = topic_model.get_topic_info().copy()
n_before = len([t for t in before_info["Topic"].tolist() if t != -1])

target_nr = max(2, n_before - 1)  # 데이터가 적으므로 1개만 줄여 본다
print(f"원래 토픽 수: {n_before}  →  목표 토픽 수: {target_nr}")

topic_model.reduce_topics(docs, nr_topics=target_nr)

after_info = topic_model.get_topic_info()
n_after = len([t for t in after_info["Topic"].tolist() if t != -1])
print(f"축소 후 토픽 수: {n_after}")

display(after_info)

reduced_keywords = {
    int(t): [w for w, _ in topic_model.get_topic(t)[:10]]
    for t in topic_model.get_topics().keys() if t != -1
}
with open(OUTPUT_DIR / "ch8B_reduced_topic_keywords.json", "w", encoding="utf-8") as f:
    json.dump(reduced_keywords, f, ensure_ascii=False, indent=2)
print(f"저장 완료: {OUTPUT_DIR / 'ch8B_reduced_topic_keywords.json'}")

# 축소 후 평가지표 재계산
post_npmi = npmi_coherence(topic_model, docs)
post_div = topic_diversity(topic_model)
print(f"\n축소 후 NPMI            : {post_npmi:+.4f}  (이전 {eval_npmi:+.4f})")
print(f"축소 후 Topic Diversity : {post_div:.4f}  (이전 {eval_div:.4f})")


## 12. Optuna 다목적 최적화 — 일관성↑ × 노이즈↓

UMAP·HDBSCAN의 파라미터를 베이지안 최적화로 자동 탐색한다. 두 목표(일관성 최대화, 노이즈 비율 최소화)는 서로 충돌하므로 결과는 단일 최적 해가 아니라 **파레토 프론트** — trade-off의 후보들 — 가 나온다.

> **강의 데모**: `n_trials=10`. 실무에서는 50~100회. 이 데이터셋이 작아 일부 trial은 1개 토픽만 만드는 식으로 실패할 수 있고, 그런 trial은 NPMI=0으로 떨어져 자동으로 후순위가 된다.


In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    n_neighbors = trial.suggest_int("n_neighbors", 3, 15)
    n_components = trial.suggest_int("n_components", 2, 8)
    min_cluster_size = trial.suggest_int("min_cluster_size", 3, 12)
    min_samples = trial.suggest_int("min_samples", 1, 5)

    umap_t = UMAP(
        n_neighbors=n_neighbors, n_components=n_components,
        min_dist=0.0, metric="cosine", random_state=SEED,
    )
    hdbscan_t = HDBSCAN(
        min_cluster_size=min_cluster_size, min_samples=min_samples,
        metric="euclidean", cluster_selection_method="eom",
    )
    vectorizer_t = CountVectorizer(
        token_pattern=r"(?u)[가-힣A-Za-z0-9]{2,}", min_df=1, ngram_range=(1, 1),
    )
    model_t = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_t, hdbscan_model=hdbscan_t,
        vectorizer_model=vectorizer_t, language="multilingual",
        calculate_probabilities=False, verbose=False,
    )
    try:
        topics_t, _ = model_t.fit_transform(docs, embeddings=embeddings)
    except Exception:
        return 0.0, 1.0

    n_t = len([t for t in set(topics_t) if t != -1])
    if n_t < 2:
        return 0.0, 1.0  # 토픽이 1개면 의미 있는 평가 불가

    coh = npmi_coherence(model_t, docs)
    noise_ratio = sum(1 for t in topics_t if t == -1) / len(topics_t)
    return coh, noise_ratio


study = optuna.create_study(directions=["maximize", "minimize"])
study.optimize(objective, n_trials=10, show_progress_bar=False)

# 파레토 프론트 출력
pareto_rows = []
for t in study.best_trials:
    pareto_rows.append({
        "trial": t.number,
        "NPMI": round(t.values[0], 4),
        "noise_ratio": round(t.values[1], 4),
        **t.params,
    })
pareto_df = pd.DataFrame(pareto_rows).sort_values(["NPMI", "noise_ratio"], ascending=[False, True])
display(pareto_df)
pareto_df.to_csv(OUTPUT_DIR / "ch8B_optuna_pareto.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_DIR / 'ch8B_optuna_pareto.csv'}")

print("\n[해석 가이드]")
print("- NPMI가 높고 노이즈 비율도 낮은 trial이 이상적이지만, 보통은 trade-off가 존재한다.")
print("- 자신의 분석 목적(일관성 우선 vs 데이터 손실 최소화)에 맞춰 한 trial을 선택한다.")


## 13. 토픽 간 시계열 상관 — 함께 움직이는 주제

`topics_over_time` 결과(7절에서 만든 `topics_over_time` 변수)를 피벗해 토픽×시간 행렬을 만들고 상관 행렬을 계산한다. 절대값이 큰 쌍은 함께 움직이는 주제다.


In [ ]:
# 원래 5개 토픽 기준의 동적 결과를 사용한다.
# 11절 reduce_topics() 실습이 topic_model의 내부 상태를 바꾸더라도,
# 상관 분석은 7절에서 저장한 topics_over_time을 기준으로 해석한다.
topics_over_time2 = topics_over_time.copy()

pivot_df = topics_over_time2.pivot(
    index="Timestamp", columns="Topic", values="Frequency",
).fillna(0)
correlation_matrix = pivot_df.corr().round(3)
display(correlation_matrix)
correlation_matrix.to_csv(OUTPUT_DIR / "ch8B_topic_correlation.csv", encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_DIR / 'ch8B_topic_correlation.csv'}")

# 절대값 0.5 이상의 토픽 쌍
strong_pairs = []
cols = correlation_matrix.columns.tolist()
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        v = correlation_matrix.iloc[i, j]
        if abs(v) >= 0.5:
            strong_pairs.append({
                "topic_a": int(cols[i]),
                "label_a": topic_label_guess.get(int(cols[i]), f"주제 {cols[i]}"),
                "topic_b": int(cols[j]),
                "label_b": topic_label_guess.get(int(cols[j]), f"주제 {cols[j]}"),
                "corr": float(v),
                "relation": "동행" if v > 0 else "역행",
            })

print("\n[토픽 간 시계열 상관 해석]")
print("- 상관계수는 두 토픽의 시간별 빈도가 함께 오르내리는 정도를 뜻한다.")
print("- 양수는 함께 증가/감소하는 동행 관계, 음수는 한쪽이 늘 때 다른 쪽이 줄어드는 역행 관계다.")

if strong_pairs:
    strong_pairs_df = pd.DataFrame(strong_pairs).sort_values("corr", key=lambda s: s.abs(), ascending=False)
    print("\n강한 동행/역행 토픽 쌍 (|corr| ≥ 0.5):")
    display(strong_pairs_df)
    for row in strong_pairs_df.itertuples(index=False):
        print(
            f"- {row.label_a}(토픽 {row.topic_a})와 {row.label_b}(토픽 {row.topic_b})는 "
            f"{row.relation} 관계다(corr={row.corr:+.3f})."
        )
else:
    print("- |corr| ≥ 0.5인 토픽 쌍이 없다. 이 데이터에서는 주제들이 시간상 뚜렷하게 함께 움직인다고 보기 어렵다.")
    print("- 실습에서는 데이터 기간을 늘리거나 특정 사건 시점의 문서를 더 추가하면 상관 신호가 더 뚜렷해질 수 있다.")


## 14. 토픽 생명 주기 — 출현 → 최고점 → 소멸

각 토픽이 **언제 처음 등장했고**(출현), **언제 가장 활발했고**(최고점), **언제 사라졌는지**(소멸)를 자동으로 식별한다.


In [ ]:
def analyze_topic_lifecycle(tot_df, keyword_map, threshold=0.01):
    """토픽의 출현·최고점·소멸 시점 자동 식별."""
    lifecycle = {}
    for topic in [t for t in tot_df["Topic"].unique() if t != -1]:
        td = tot_df[tot_df["Topic"] == topic].sort_values("Timestamp")
        if td.empty:
            continue
        # 출현: 빈도가 임계값을 넘는 첫 시점
        above = td[td["Frequency"] > threshold]
        emergence = above.iloc[0]["Timestamp"] if len(above) > 0 else None

        # 최고점
        peak_idx = td["Frequency"].idxmax()
        peak_t = td.loc[peak_idx, "Timestamp"]
        peak_f = float(td.loc[peak_idx, "Frequency"])

        # 소멸: 최고점 이후 임계값 밑으로 떨어진 첫 시점
        after = td.loc[peak_idx:]
        below = after[after["Frequency"] < threshold]
        decline = below.iloc[0]["Timestamp"] if len(below) > 0 else None

        lifecycle[int(topic)] = {
            "emergence": str(emergence) if emergence is not None else None,
            "peak": str(peak_t),
            "peak_frequency": peak_f,
            "decline": str(decline) if decline is not None else None,
            "keywords": keyword_map.get(int(topic), [])[:5],
        }
    return lifecycle


lifecycle = analyze_topic_lifecycle(topics_over_time2, topic_keywords, threshold=0.5)
lifecycle_rows = []
for topic_id, info in lifecycle.items():
    lifecycle_rows.append({
        "topic": topic_id,
        "estimated_label": topic_label_guess.get(int(topic_id), f"주제 {topic_id}"),
        "keywords": ", ".join(info["keywords"]),
        "emergence": info["emergence"],
        "peak": info["peak"],
        "peak_frequency": info["peak_frequency"],
        "decline": info["decline"],
    })
lifecycle_df = pd.DataFrame(lifecycle_rows).sort_values(["peak", "topic"])
display(lifecycle_df)
lifecycle_df.to_csv(OUTPUT_DIR / "ch8B_topic_lifecycle.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_DIR / 'ch8B_topic_lifecycle.csv'}")

print("\n[토픽 생명 주기 해석]")
print("- emergence는 토픽이 처음 관측된 시점, peak는 가장 많이 등장한 시점, decline은 최고점 이후 빈도가 낮아진 첫 시점이다.")
if lifecycle_rows:
    for row in lifecycle_rows:
        decline_text = row["decline"] if row["decline"] is not None else "분석 기간 안에서는 뚜렷하지 않음"
        print(
            f"- {row['estimated_label']}(토픽 {row['topic']})은 {row['emergence']}에 관측되기 시작했고, "
            f"{row['peak']}에 최고점({row['peak_frequency']:.0f}건)을 보였다. "
            f"소멸/하락 시점: {decline_text}."
        )
else:
    print("- 해석할 유효 토픽이 없다.")

print("\n[해석 가이드]")
print("- emergence가 분석 시작 직후이고 decline이 비어 있으면: 안정적인 상시 토픽")
print("- emergence가 늦고 peak가 emergence 직후면: 급상승 패턴")
print("- emergence/peak는 있는데 decline이 비어 있으면: 점진 상승 중")


## 15. 멀티모달 토픽 모델링 — CLIP (텍스트 + 이미지)

`data/input/ch8B_multimodal_data.csv`에 20개 패션 상품(드레스·재킷·신발·가방)의 **영문 캡션**이 있고, `data/input/multimodal_images/`에 대응하는 PNG 이미지 20장이 있다. CLIP 모델은 텍스트와 이미지를 같은 512차원 공간에 투영하므로, 캡션과 이미지의 의미적 유사성을 동시에 활용하여 토픽을 발견한다.

> **첫 실행 시 약 600MB 모델(`clip-ViT-B-32`)을 다운로드**한다. `bertopic[vision]`이 설치되어 있어야 한다 — 미설치라면 셀이 자동으로 안내 메시지를 출력하고 통과한다. 무거운 다운로드를 원치 않으면 `RUN_MULTIMODAL = False`로 둔다.


In [ ]:
# === 멀티모달 시연 (CLIP + chapter5 패션 데이터) ===
RUN_MULTIMODAL = True  # False로 두면 다운로드/실행을 건너뛴다

multimodal_status = "skipped"
multimodal_reason = ""

if not RUN_MULTIMODAL:
    multimodal_reason = "RUN_MULTIMODAL = False (사용자 설정). 실행하려면 True로 바꾸세요."
else:
    try:
        from bertopic.backend import MultiModalBackend
        from bertopic.representation import VisualRepresentation
    except ImportError as exc:
        multimodal_reason = (
            f"의존성 누락: {exc}. 다음을 실행한 뒤 다시 시도하세요: "
            f"`pip install \"bertopic[vision]\" pillow`"
        )
    else:
        mm_csv = BASE_DIR / "data" / "input" / "ch8B_multimodal_data.csv"
        mm_img_root = BASE_DIR / "data" / "input"

        if not mm_csv.exists():
            multimodal_reason = f"데이터 파일이 없습니다: {mm_csv}"
        else:
            mm_df = pd.read_csv(mm_csv)
            captions = mm_df["caption"].tolist()
            image_paths = [str(mm_img_root / p) for p in mm_df["image_path"].tolist()]

            missing_imgs = [p for p in image_paths if not Path(p).exists()]
            if missing_imgs:
                multimodal_reason = f"이미지 누락 {len(missing_imgs)}개. 첫 사례: {missing_imgs[0]}"
            else:
                print(f"로드된 캡션: {len(captions)}, 이미지: {len(image_paths)}")
                print("CLIP(clip-ViT-B-32) 로드 중... 첫 실행 시 약 600MB 다운로드.")

                mm_emb = MultiModalBackend("clip-ViT-B-32", batch_size=16)
                mm_repr = {"Visual_Aspect": VisualRepresentation()}
                mm_model = BERTopic(
                    embedding_model=mm_emb,
                    representation_model=mm_repr,
                    min_topic_size=3,  # 데이터가 20개로 작으므로 낮춤
                    verbose=False,
                )

                mm_topics, _ = mm_model.fit_transform(documents=captions, images=image_paths)

                mm_info = mm_model.get_topic_info()
                print("\n[멀티모달 토픽 분석 결과]")
                display(mm_info[["Topic", "Count", "Name", "Representation"]])

                # 토픽별 키워드 + 대표 이미지 캡션
                mm_topic_rows = []
                for tid in sorted(t for t in set(mm_topics) if t != -1):
                    words = [w for w, _ in mm_model.get_topic(tid)[:5]]
                    docs_in_topic = [captions[i] for i, t in enumerate(mm_topics) if t == tid]
                    mm_topic_rows.append({
                        "topic": int(tid),
                        "keywords": ", ".join(words),
                        "n_items": len(docs_in_topic),
                        "sample_caption": docs_in_topic[0] if docs_in_topic else "",
                    })
                mm_summary = pd.DataFrame(mm_topic_rows)
                display(mm_summary)

                mm_summary.to_csv(OUTPUT_DIR / "ch8B_multimodal_topic_info.csv", index=False, encoding="utf-8-sig")
                print(f"\n저장 완료: {OUTPUT_DIR / 'ch8B_multimodal_topic_info.csv'}")
                multimodal_status = "ok"

print(f"\n[multimodal] status={multimodal_status} | {multimodal_reason}")
